# DiCo-NLI runner (Kaggle)

Đang chạy: **thí nghiệm tăng dữ liệu bằng bắc cầu** (`augment_transitive=true`, `src/dico/augment.py`): Qwen EN/ES/EU đơn ngữ, 3 seed + OOF 3 fold, tên run `qwen_cls_{en,es,eu}_aug`. Chạy song song được với `runner.ipynb` (notebook lưu model) ở một phiên khác.

Notebook mỏng: toàn bộ logic nằm trong repo (`scripts/kaggle_queue.sh`).

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**. Chạy bằng *Save Version → Save & Run All (Commit)* để không phải mở tab.

Hai GPU chạy song song hai hàng đợi (mỗi GPU một process, giữ nguyên batch size của config):
- GPU 0: Qwen EN/ES/EU _aug, 3 seed mỗi ngôn ngữ
- GPU 1: OOF 3 fold cho EN/ES/EU _aug

Kết quả: tải `runs.zip` ở tab *Output*, bỏ vào `kaggle/` (gồm `qwen_cls_*_aug/seed*/` và `oof/qwen_cls_*_aug/`).


In [ ]:
REPO_URL = 'https://github.com/danh-tc/DiCo-NLI.git'
!git clone --depth 1 $REPO_URL /kaggle/working/dico-nli
%cd /kaggle/working/dico-nli
!git log -1 --oneline

In [ ]:
!pip install -q -e . peft
# peft mới đòi torchao >= 0.16, Kaggle cài sẵn 0.10 -> gỡ torchao (code không dùng nó)
!pip uninstall -y -q torchao
!python -c "import peft; from peft.import_utils import is_torchao_available as f; print('peft', peft.__version__, 'torchao', f())"
!bash scripts/setup_data.sh

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import sys; sys.path.insert(0, 'src'); from dico import env; print(env.describe())"
# Tải model một lần trước, để hai hàng đợi không tải song song
!python -c "from huggingface_hub import snapshot_download as d; d('Qwen/Qwen2.5-1.5B')"

In [ ]:
%%bash
# Kiểm tra nhanh trước (khoảng 2–3 phút, gồm lưu và nạp lại model): 0.02 epoch Qwen EN + augment; lỗi thì dừng, không tốn cả hàng đợi
set -e
CUDA_VISIBLE_DEVICES=0 PYTHONPATH=src python -m dico.train --config configs/qwen_cls_en.yaml --seed 1 --set name=smoke epochs=0.02 "decoders=[argmax,self_swap]" augment_transitive=true keep_model=true > /kaggle/working/smoke.log 2>&1 || { tail -40 /kaggle/working/smoke.log; exit 1; }
# nạp lại model vừa lưu, so với probs lúc train
PYTHONPATH=src python -c "
from pathlib import Path; import numpy as np
from dico.data import load_split; from dico.predict import predict_run
r = Path('/kaggle/working/runs/smoke/seed1'); [(p, s)] = predict_run(r, [load_split('dev', [1])])
ref = np.load(r / 'dev_track1_probs.npy'); print('reload max |dp|', np.abs(p - ref).max(), 'argmax agree', (p.argmax(1) == ref.argmax(1)).mean())
assert (p.argmax(1) == ref.argmax(1)).mean() > 0.99
" > /kaggle/working/smoke_reload.log 2>&1 || { tail -30 /kaggle/working/smoke_reload.log; exit 1; }
grep "reload max" /kaggle/working/smoke_reload.log
rm -rf /kaggle/working/runs/smoke
set +e
echo smoke OK
# Hai hàng đợi song song, mỗi GPU một hàng; xong cả hai mới sang ô sau
export SUFFIX=_aug EXTRA="augment_transitive=true"
bash scripts/kaggle_queue.sh 0 train:en:1 train:en:2 train:en:3 train:es:1 train:es:2 train:es:3 train:eu:1 train:eu:2 train:eu:3 &
bash scripts/kaggle_queue.sh 1 oof:en oof:es oof:eu &
wait
cat /kaggle/working/runs/logs/status.txt

In [ ]:
# Nếu có FAIL: xem 40 dòng cuối log của job đó
!grep FAIL /kaggle/working/runs/logs/status.txt || echo "no failures"
!for f in $(grep FAIL /kaggle/working/runs/logs/status.txt | awk '{print $3}'); do echo "== $f"; tail -40 /kaggle/working/runs/logs/$f.log; done

In [ ]:
!cd /kaggle/working && zip -qr runs.zip runs -x '*/ckpt/*' '*/model/*' && ls -lh runs.zip